
##  Dry and Moist sumulations in Jovian EZ zone 

- Note: restart the jupyter kernel before rerunning, otherwise you will get a runtime error

In [1]:
#! /usr/bin/env python3
import numpy as np
import sys
import h5py

sys.path.append("../python")  # Adjust the path to build/python/
sys.path.append(".")

## load the necessary modules
from canoe import def_species, load_configure, index_map
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
from canoe.harp import radiation_band, radiation


In [2]:
# load the configuration file
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")
pin.set_boolean("job", "verbose", False)  

# register the speicies and thermodynamics
vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

# load bands and opacities
config = load_configure("juno_mwr.yaml")

Log, "2025-06-14 21:32:37",        canoe, 1., "Installing monitor canoe"
Log, "2025-06-14 21:32:37",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-06-14 21:32:37",         snap, 3., "Installing monitor snap"
Log, "2025-06-14 21:32:37",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-06-14 21:32:37",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-06-14 21:32:37",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-06-14 21:32:37",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"


In [3]:
# set incident angles 
angle =[0.0, 15, 30.0, 45.]  # in degrees
angles = ' '.join([f"({x},)" for x in angle])
pin.set_string("radiation","outdir",angles)

print(pin.get_string("radiation","outdir"))

(0.0,) (15,) (30.0,) (45.0,)


In [4]:
# set the gravity 
pin.set_string("hydro", "grav_acc1", f"{-23.3}") # Equatorial zone
# pin.set_string("hydro", "grav_acc1", f"{-27.01}") # pole 

print("Gravity acceleration:", pin.get_string("hydro", "grav_acc1"))

Gravity acceleration: -23.3


In [5]:
# get index_map 
pindex = index_map.get_instance()
iNH3 = pindex.get_vapor_id("NH3")
iH2O = pindex.get_vapor_id("H2O")

print(f"iNH3 = {iNH3}, iH2O = {iH2O}")

P0 = pin.get_real("mesh", "ReferencePressure")
print(f"Reference Pressure: {P0} Pa")

iNH3 = 2, iH2O = 1
Reference Pressure: 100000.0 Pa


In [6]:
# set mesh; we use the 1st column of the 1st meshblock of the Athena++ mesh

nx2 = 1  # air column number, single column in this case
pin.set_string("mesh", "nx2", f"{nx2}")

# init the Mesh
mesh = Mesh(pin) 
mesh.initialize(pin)

# get the first meshblock
mb = mesh.meshblock(0)  

Log, "2025-06-14 21:32:37",         snap, 4.1., "Initialize Decomposition"
Log, "2025-06-14 21:32:37",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-06-14 21:32:37", microphysics, 7., "Installing monitor microphysics"
Log, "2025-06-14 21:32:37", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-06-14 21:32:37",         harp, 9., "Installing monitor harp"
Log, "2025-06-14 21:32:37",         harp, 9.1., "Initialize Radiation"
Log, "2025-06-14 21:32:37",         harp, 9.1.1., "Load Radiation bands from juno_mwr.yaml"
Log, "2025-06-14 21:32:37",         harp, 9.1.1.1., "Initialize RadiationBand CH1"
Log, "2025-06-14 21:32:37",      opacity, 9.1.1.2., "Installing monitor opacity"
Log, "2025-06-14 21:32:37",      opacity, 9.1.1.2.1., "Create Absorber CIA"
Log, "2025-06-14 21:32:37",      opacity, 9.1.1.3.1., "Create Absorber NH3"
Log, "2025-06-14 21:32:37",      opacity, 9.1.1.4.1., "Create Absorber H2O"
Log, "2025-06-14 21:32:37",      opacity, 9.1.1.5.1., "Create Absorb

In [7]:
## set EZ parameters

##  EZ abundances and temperature in Cheng (2020) 
xNH3=351     # deep-layer abundance, ppmv
xH2O=2500    # ppmv
T1bar=169    # 1-bar temperature, K

## use the first ap column of the block
Jindex=0   # Jindex ∈ [0, nx2-1]

## RH limit 
RHmax=1  # limit of relative humidity, 1 means 100%, only affects NH3 cloud-layer, set to 1 in EZ
maxint=200  # do not change


In [8]:

## ---------------    a moist adiabatic + uniform NH3  ---------------------------
adiabate="pseudo"

# build the atmosphere with a fixed top temperature (T1bar) 
mb.construct_atmosphere(pin, xNH3, T1bar, RHmax, Jindex, adiabate, xH2O, maxint)

## calc for the Jindex_st column profile
aircolumn = mb.get_aircolumn(mb.k_st, mb.j_st + Jindex, mb.i_st, mb.i_ed)   
#### loop over layers bottom up
nlyr=len(aircolumn)             ## default 1600 pressure layers


# get T, P, and θ for each layer
for i in range(nlyr): 
    ap= aircolumn[i].to_mole_fraction()  # air column profile for layer i
    p = ap.get_pressure()  # pressure in Pa
    T =  ap.get_temp()  # temperature in K
    theta = mb.get_theta(P0, mb.k_st, mb.j_st + Jindex,mb.i_st + i)
    print(f"Layer {i}: P = {p*1E-5:.2f} bar, T = {T:.2f} K, θ = {theta:.2f} K")


Log, "2025-06-14 21:32:37", pycanoe_construct_atmosphere, 18., "Installing monitor pycanoe_construct_atmosphere"
Layer 0: P = 8103.08 bar, T = 2253.14 K, θ = 157.40 K
Layer 1: P = 8024.13 bar, T = 2247.64 K, θ = 157.47 K
Layer 2: P = 7945.94 bar, T = 2242.15 K, θ = 157.54 K
Layer 3: P = 7868.52 bar, T = 2236.68 K, θ = 157.62 K
Layer 4: P = 7791.85 bar, T = 2231.21 K, θ = 157.69 K
Layer 5: P = 7715.93 bar, T = 2225.75 K, θ = 157.76 K
Layer 6: P = 7640.74 bar, T = 2220.31 K, θ = 157.83 K
Layer 7: P = 7566.29 bar, T = 2214.87 K, θ = 157.90 K
Layer 8: P = 7492.57 bar, T = 2209.45 K, θ = 157.97 K
Layer 9: P = 7419.56 bar, T = 2204.04 K, θ = 158.04 K
Layer 10: P = 7347.26 bar, T = 2198.64 K, θ = 158.11 K
Layer 11: P = 7275.67 bar, T = 2193.25 K, θ = 158.18 K
Layer 12: P = 7204.78 bar, T = 2187.87 K, θ = 158.25 K
Layer 13: P = 7134.58 bar, T = 2182.50 K, θ = 158.32 K
Layer 14: P = 7065.06 bar, T = 2177.14 K, θ = 158.38 K
Layer 15: P = 6996.22 bar, T = 2171.79 K, θ = 158.45 K
Layer 16: P = 692

In [9]:
## demo set T for given layer
ilayer = 100  # layer index

aircolumn = mb.get_aircolumn(mb.k_st, mb.j_st + Jindex, mb.i_st, mb.i_ed)  
T = mb.get_temp(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the new temperature
P = mb.get_pressure(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the pressure for this layer
print(f"old: Layer {ilayer}: P = {P:.2f} bar, T = {T:.2f} K")


ap_mole = aircolumn[ilayer].to_mole_fraction() 
P = ap_mole.get_pressure()  # get the pressure for this layer 
T = ap_mole.get_temp()  # get the temperature for this layer
print(f"airparcel: P = {P:.2f} bar, T = {T} K")

# set the temperature for this layer
T_new = 2000.0  # set to 2000 K
ap_mole.set_temp(T_new)  # set the new temperature

print("airparcel: P=", ap_mole.get_pressure()," T=",ap_mole.get_temp())


# distribute the airparcel to the aircolumn
mb.distribute_to_primitive(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer, ap_mole)


T = mb.get_temp(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the new temperature
P = mb.get_pressure(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the pressure for this layer
print(f"new: Layer {ilayer}: P = {P:.2f} Pa, T = {T} K")



old: Layer 100: P = 304371270.36 bar, T = 1754.58 K
airparcel: P = 304371270.36 bar, T = 1754.5837872288187 K
airparcel: P= 304371270.3645299  T= 2000.0
new: Layer 100: P = 304371270.36 Pa, T = 2000.0 K


In [10]:
## demo to overwrite the NH3,H2O  in mass fraction kg/kg

ilayer = 100  # layer index to overwrite, e.g., 100th layer

NH3_kg=0.0027  # kg/kg
# H2O_desired=0.018  # kg/kg

ap_mass = aircolumn[ilayer].to_mass_fraction()  # air column profile for layer i in mass fraction
print(ap_mass.hydro()[iNH3])

ap_mass.set_property(iNH3, NH3_kg) 
mb.distribute_to_primitive(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer, ap_mass)

print(ap_mass.hydro()[iNH3])


0.0025262302127678398
0.0027


In [11]:
## demo to overwrite the NH3,H2O in ppmv

ilayer = 100  # layer index to overwrite, e.g., 100th layer

NH3_ppm=381  # ppmv
# H2O_ppm=4000  # ppmv

ap_mole = aircolumn[ilayer].to_mole_fraction()
print(ap_mole.hydro()[iNH3])

ap_mole.set_property(iNH3, NH3_ppm/1E6) 
mb.distribute_to_primitive(mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer, ap_mole)  

print(ap_mole.hydro()[iNH3])



0.0003752013454543327
0.000381


In [ ]:
ilayer = 100  # layer index to overwrite, e.g., 100th layer

ielec = pindex.get_tracer_id("e-")  # index for electron
iNa = pindex.get_tracer_id("Na")  # index for sodium

# print(ielec)
# print(iNa)

elec=mb.get_tracer(ielec, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the electron tracer for this layer
Na=mb.get_tracer(iNa, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer)  # get the sodium tracer for this layer
print(elec)
print(Na)

mb.set_tracer_layer(ielec, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer, 1.0)  # set the electron tracer to 1.0
mb.set_tracer_layer(iNa, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer, 0.1)  # set the sodium tracer to 0.1

print(mb.get_tracer(ielec, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer))  # get the electron tracer for this layer
print(mb.get_tracer(iNa, mb.k_st, mb.j_st + Jindex, mb.i_st + ilayer))  # get the sodium tracer for this layer


1.7484814036817826e+16
4.9881135193300074e+20
1.0
0.1
